# Lab 1 · Part A — Kaggle (All-in-One)

A condensed, run-top-to-bottom version of the starter — used to show you can **import** a notebook into Kaggle (**File → Import Notebook**), not only fork one.

Prereqs (attach first): GPU on; Mistral-7B added from **Kaggle Models**; **repeng-offline-wheels** and the **theme** dataset added as Inputs; `HF_TOKEN` in **Secrets** if you pull from HF.

In [ ]:
# --no-index = never touch PyPI; --find-links points pip at the mounted wheels.
!pip install --no-index --find-links=/kaggle/input/repeng-offline-wheels repeng
import repeng
from repeng import ControlVector, ControlModel, DatasetEntry
print("repeng imported OK")

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from repeng import ControlVector, ControlModel, DatasetEntry

# 🛠 same Kaggle Models path as A3:
MODEL_PATH = "/kaggle/input/mistral/pytorch/7b-instruct-v0.1/1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
tokenizer.pad_token_id = 0
# Default: fp16 Mistral-7B on ONE GPU (~14 GB — usually fits a single 16 GB T4, but tight).
model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, torch_dtype=torch.float16).to("cuda:0")
# --- If you hit CUDA OOM, replace the line above with ONE of these (see assignment troubleshooting): ---
#   (1) 8-bit Mistral (~7 GB, one T4; needs bitsandbytes) — the "smaller Mistral" option:
#       model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, load_in_8bit=True, device_map={"": 0})
#   (2) a smaller ungated model, e.g. Qwen/Qwen2.5-1.5B-Instruct — see the assignment's "smaller model" note.
# (Using BOTH T4s via device_map="auto" is the Bonus exercise — it's finicky on Kaggle.)
model = ControlModel(model, list(range(-5, -18, -1)))
user_tag, asst_tag = "[INST]", "[/INST]"

# 🛠 pick ONE theme (add it as a Kaggle Dataset, or upload the lab's datasets/ folder):
THEME_PATH  = "/kaggle/input/repeng-lab-themes/playful_vs_serious.json"
SUFFIX_PATH = "/kaggle/input/repeng-lab-themes/all_truncated_outputs.json"

theme = json.load(open(THEME_PATH))
output_suffixes = json.load(open(SUFFIX_PATH))

# keep training fast for the lab: use a subset of the suffix corpus
truncated = [
    tokenizer.convert_tokens_to_string(tokens[:i])
    for tokens in (tokenizer.tokenize(s) for s in output_suffixes[:256])
    for i in range(1, len(tokens))
]

def make_dataset(template, positive_personas, negative_personas, suffixes):
    ds = []
    for suffix in suffixes:
        for p, n in zip(positive_personas, negative_personas):
            ds.append(DatasetEntry(
                positive=f"{user_tag} {template.format(persona=p)} {asst_tag} {suffix}",
                negative=f"{user_tag} {template.format(persona=n)} {asst_tag} {suffix}"))
    return ds

dataset = make_dataset(theme["template"], theme["positive_personas"],
                       theme["negative_personas"], truncated)
model.reset()
vector = ControlVector.train(model, tokenizer, dataset)
print("Trained a control vector for theme:", theme["theme"])

In [ ]:
def generate_with_vector(prompt, vector, coeffs=(1.5, -1.5), max_new_tokens=100):
    pos, neg = coeffs
    if user_tag not in prompt:
        prompt = f"{user_tag} {prompt.strip()} {asst_tag}"
    ids = tokenizer(prompt, return_tensors="pt").to(model.device)
    st = dict(pad_token_id=tokenizer.eos_token_id, do_sample=False,
              max_new_tokens=max_new_tokens, repetition_penalty=1.1)
    model.reset()
    print("== baseline ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True), "\n")
    model.set_control(vector, pos)
    print("== + control ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True), "\n")
    model.set_control(vector, neg)
    print("== - control ==\n", tokenizer.decode(model.generate(**ids, **st)[0], skip_special_tokens=True))
    model.reset()

generate_with_vector(theme.get("suggested_prompt", "Give me a one-sentence pitch for a TV show."), vector)

In [ ]:
# export_gguf uses the `gguf` library — the second wheel in the offline set.
vector.export_gguf("/kaggle/working/control_vector.gguf")
import os
print("saved:", [f for f in os.listdir("/kaggle/working") if f.endswith(".gguf")])

Save Version → Commit to persist `control_vector.gguf` under the Output tab. Done.